In [ ]:
import json
import pandas as pd

In [ ]:
def load_caption_data(paraphrase, parquet_file):

    df = pd.read_parquet(parquet_file,     
                            engine="pyarrow",
                            dtype_backend="pyarrow"
    )

    df["interactionTypeNameNormal"] = (
        df["interactionTypeName"]
        .astype("string[python]")
        .str.replace(r"([a-z])([A-Z])", r"\1 \2", regex=True)
        .str.lower()
        .str.strip()
    )

    reverse = {
        "visits": "visited by",
        "visits flowers of": "flowers visited by",
        "eats": "eaten by",
        "preys on": "preyed by",
        "has host": "hosts",
        "parasite of": "infected by",
        "pathogen of": "infected by",
        "parasitoid of": "infected by",
        "interacts with": "interacting with",
    }

    target = {
        "visits": "being visited by",
        "visits flowers of": "flowers being visited by",
        "eats": "being eaten by",
        "preys on": "being preyed on by",
        "has host": "hosting",
        "parasite of": "being infected by",
        "pathogen of": "being infected by",
        "parasitoid of": "being infected by",
        "interacts with": "interacting with",
    }

    df["interactionTypeNameNormalReversed"] = df["interactionTypeNameNormal"].map(reverse)

    df["interactionTypeNameNormalTarget"] = df["interactionTypeNameNormal"].map(target)

    if paraphrase == "correct":
        df["caption"] = "Does this image show "+df["sourceTaxonName"] + " " + df["interactionTypeNameNormal"] + " " + df["targetTaxonName"]+ "? Answer with Yes or No and nothing else."

    if paraphrase == "wrong":
        df["caption"] = "Does this image show "+df["targetTaxonName"] + " " + df["interactionTypeNameNormal"] + " " + df["sourceTaxonName"] + "? Answer with Yes or No and nothing else."

    if paraphrase == "passive":
        df["caption"] = "Does this image show "+df["targetTaxonName"] + " " + df["interactionTypeNameNormalReversed"] + " " + df["sourceTaxonName"] + "? Answer with Yes or No and nothing else."

    if paraphrase == "wrong_passive":
        df["caption"] = "Does this image show "+df["sourceTaxonName"] + " " + df["interactionTypeNameNormalReversed"] + " " + df["sourceTaxonName"] + "? Answer with Yes or No and nothing else."

    if paraphrase == "target":
        df["caption"] = "Does this image show "+df["targetTaxonName"] + " " + df["interactionTypeNameNormalReversed"] + " with another organism" + "? Answer with Yes or No and nothing else."

    if paraphrase == "source":
        df["caption"] = "Does this image show "+df["sourceTaxonName"] + " " + df["interactionTypeNameNormal"] + " with another organism" + "? Answer with Yes or No and nothing else."

    if paraphrase == "no_relation":
        df["caption"] = "Does this image show "+df["sourceTaxonName"] + " and " + df["targetTaxonName"] + "? Answer with Yes or No and nothing else."

    df["fileName"] = df["image_id"] + df["file_ext"]

    return df[["caption", "fileName"]]

In [ ]:
input_file = "icun.parquet"

PARAPHRASES = [
    "correct",
    "wrong",
    "passive",
    "wrong_passive",
    "target",
    "source",
    "no_relation",
]

for paraphrase in PARAPHRASES:
    df = load_caption_data(paraphrase, input_file)